# 90 — Live pipeline runner

Run the live Bronze-to-Gold sequence in one controlled order. Each child notebook remains independently runnable for recovery.


In [ ]:
NOTEBOOK_TIMEOUT_SECONDS = 1800
STOP_ON_ERROR = True
JOB_RUN_ID = ""  # Optional caller-supplied ID; generated when blank.

LIVE_STEPS = [
    ("00_setup_cfg", {}),
    ("01_bronze_get_latest", {}),
    ("01a_cfg_schema_capture_live", {}),
    ("02_silver_formatter", {}),
    ("03_silver_business_rules", {}),
    ("04_gold_model", {}),
    ("05_gold_dimensions", {}),
]


In [ ]:
import uuid
from datetime import datetime

from delta.tables import DeltaTable
from notebookutils import mssparkutils

PIPELINE_NAME = "90_run_live_pipeline"
JOB_RUN_ID = JOB_RUN_ID or str(uuid.uuid4())
started_at = datetime.utcnow()
results = []


def merge_monitor_row(table_name, row, schema, condition):
    source = spark.createDataFrame([row], schema)
    target = DeltaTable.forName(spark, table_name)
    (target.alias("target").merge(source.alias("source"), condition)
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())


def record_step(step_sequence, notebook_name, status, step_started,
                child_result=None, error_message=None):
    merge_monitor_row(
        "monitoring.cfg_job_step_run",
        (JOB_RUN_ID, step_sequence, notebook_name, step_started,
         None if status == "RUNNING" else datetime.utcnow(), status,
         child_result[:4000] if child_result else None,
         error_message[:4000] if error_message else None, datetime.utcnow()),
        "job_run_id string,step_sequence int,notebook_name string,started_at timestamp,ended_at timestamp,status string,child_result string,error_message string,last_updated_at timestamp",
        "target.job_run_id = source.job_run_id AND target.step_sequence = source.step_sequence",
    )


# Setup is deliberately first: it creates or upgrades the two orchestration
# monitor tables before this runner writes its first status record.
setup_name, setup_parameters = LIVE_STEPS[0]
setup_started = datetime.utcnow()
print(f"JOB_RUN_ID={JOB_RUN_ID}")
print(f"=== START {setup_name} ===")
try:
    setup_result = mssparkutils.notebook.run(
        setup_name, NOTEBOOK_TIMEOUT_SECONDS,
        {**setup_parameters, "JOB_RUN_ID": JOB_RUN_ID},
    )
except Exception:
    # Monitoring tables may not yet exist, so Fabric's notebook error is the
    # authoritative failure record for a failed setup bootstrap.
    raise

merge_monitor_row(
    "monitoring.cfg_job_run",
    (JOB_RUN_ID, PIPELINE_NAME, started_at, None, "RUNNING", 1, 0, None,
     datetime.utcnow()),
    "job_run_id string,pipeline_name string,started_at timestamp,ended_at timestamp,status string,steps_succeeded int,steps_failed int,error_message string,last_updated_at timestamp",
    "target.job_run_id = source.job_run_id",
)
record_step(1, setup_name, "SUCCESS", setup_started, str(setup_result))
results.append((setup_name, "SUCCESS", str(setup_result)))
print(f"=== SUCCESS {setup_name} ===")

try:
    for step_sequence, (notebook_name, parameters) in enumerate(LIVE_STEPS[1:], start=2):
        step_started = datetime.utcnow()
        print(f"=== START {notebook_name}; JOB_RUN_ID={JOB_RUN_ID} ===")
        record_step(step_sequence, notebook_name, "RUNNING", step_started)
        try:
            result = mssparkutils.notebook.run(
                notebook_name, NOTEBOOK_TIMEOUT_SECONDS,
                {**parameters, "JOB_RUN_ID": JOB_RUN_ID},
            )
            result_text = str(result)
            record_step(step_sequence, notebook_name, "SUCCESS", step_started,
                        child_result=result_text)
            results.append((notebook_name, "SUCCESS", result_text))
            print(f"=== SUCCESS {notebook_name} ===")
        except Exception as exc:
            error_text = str(exc)[:4000]
            record_step(step_sequence, notebook_name, "FAILED", step_started,
                        error_message=error_text)
            results.append((notebook_name, "FAILED", error_text))
            print(f"=== FAILED {notebook_name}: {error_text} ===")
            if STOP_ON_ERROR:
                raise
finally:
    failed = [name for name, status, _ in results if status == "FAILED"]
    succeeded = sum(1 for _, status, _ in results if status == "SUCCESS")
    status = "FAILED" if failed else "SUCCESS"
    error_message = f"Failed notebook(s): {failed}" if failed else None
    merge_monitor_row(
        "monitoring.cfg_job_run",
        (JOB_RUN_ID, PIPELINE_NAME, started_at, datetime.utcnow(), status,
         succeeded, len(failed), error_message, datetime.utcnow()),
        "job_run_id string,pipeline_name string,started_at timestamp,ended_at timestamp,status string,steps_succeeded int,steps_failed int,error_message string,last_updated_at timestamp",
        "target.job_run_id = source.job_run_id",
    )

failed = [name for name, status, _ in results if status == "FAILED"]
print(
    f"Live pipeline {status}; JOB_RUN_ID={JOB_RUN_ID}; "
    f"started={started_at.isoformat()}; results={results}"
)
if failed:
    raise RuntimeError(f"Live pipeline failed notebook(s): {failed}")
